In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow import keras
from tensorflow.keras import layers

# ==========================
# Load Dataset
# ==========================

df = pd.read_csv("loan.csv")

# ==========================
# Remove Unnecessary Column
# ==========================

df.drop("Loan_ID", axis=1, inplace=True)

# ==========================
# Handle Missing Values
# ==========================

categorical_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Self_Employed'
]

for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

numerical_cols = [
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

# ==========================
# Convert Target Column
# ==========================

df['Loan_Status'] = df['Loan_Status'].map({
    'Y': 1,
    'N': 0
})

# ==========================
# Separate X and y
# ==========================

X = df.drop('Loan_Status', axis=1)

y = df['Loan_Status']

# ==========================
# One Hot Encoding
# ==========================

X = pd.get_dummies(
    X,
    columns=[
        'Gender',
        'Married',
        'Education',
        'Self_Employed',
        'Property_Area'
    ],
    drop_first=True
)

# ==========================
# Handle Dependents Column
# ==========================

X['Dependents'] = X['Dependents'].replace('3+', 3)
X['Dependents'] = X['Dependents'].astype(int)

# ==========================
# Train Test Split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# Feature Scaling
# ==========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# Build ANN Model
# ==========================

model = keras.Sequential([

    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(8, activation='relu'),

    layers.Dense(4, activation='relu'),

    layers.Dense(1, activation='sigmoid')
])

# ==========================
# Compile Model
# ==========================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==========================
# Train Model
# ==========================

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_split=0.2
)

# ==========================
# Evaluate Model
# ==========================

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# ==========================
# Predict Sample Customer
# ==========================

sample = X_test[:1]

prediction = model.predict(sample)

print("Probability:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Loan Approved")
else:
    print("Loan Rejected")

/tmp/ipykernel_1747/2411671627.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipykernel_1747/2411671627.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.5944 - loss: 0.7614 - val_accuracy: 0.5556 - val_loss: 0.7442
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5969 - loss: 0.7318 - val_accuracy: 0.5455 - val_loss: 0.7202
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6071 - loss: 0.7049 - val_accuracy: 0.5859 - val_loss: 0.6975
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6301 - loss: 0.6785 - val_accuracy: 0.6364 - val_loss: 0.6791
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6582 - loss: 0.6562 - val_accuracy: 0.6364 - val_loss: 0.6603
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6633 - loss: 0.6366 - val_accuracy: 0.6566 - val_loss: 0.6463
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6735 - loss: 0.6215 - val_accuracy: 0.6667 - val_loss: 0.6343
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6811 - loss: 0.6085 - val_accuracy: 0.6

In [4]:
model.save("loan_model.keras")

In [5]:
import joblib

model.save("loan_model.keras")

joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [6]:
print(X.columns.tolist())


['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Gender_Male', 'Married_Yes', 'Education_Not Graduate', 'Self_Employed_Yes', 'Property_Area_Semiurban', 'Property_Area_Urban']
